# NB09：多智能體系統與 LangGraph — 從單一 Agent 到協作智能體網絡

## 學習目標

| 主題 | 說明 |
|------|------|
| LangGraph 核心架構 | StateGraph、節點（nodes）、邊（edges）、條件路由 |
| Planner → Executor → Critic | 三層智能體協作模式 |
| 階層式委派 | Orchestrator 指揮多個專業子智能體 |
| Human-in-the-Loop | 加入人工審查檢查點 |
| 並行執行 | asyncio 並行子智能體，提升效率 |

## 系列位置

```
NB01 基礎 RAG → NB02 Chunking → NB03 混合搜索 → NB04 評估框架
→ NB05 Query 轉換 → NB06 進階索引 → NB07 生產優化
→ NB08 ReAct & Tool Calling → [NB09 多智能體 LangGraph] ← 你在這裡
```

---

**多智能體系統**（Multi-Agent System）將複雜任務拆解給多個專業 AI Agent 協作完成。  
**LangGraph** 提供圖形化的狀態機框架，讓你能夠精確控制 Agent 之間的資訊流動與決策邏輯。

---
## Part 0：環境設定

安裝所需套件並載入 API 金鑰。

In [ ]:
# 安裝所需套件（第一次執行時需要）
# !pip install langgraph langchain langchain-openai python-dotenv openai

In [ ]:
import os
import json
import time
import asyncio
from typing import TypedDict, Annotated, List, Optional, Literal
from dotenv import load_dotenv

# LangGraph 核心
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

# LangChain OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 載入 .env 檔案
load_dotenv()

# 初始化 LLM（使用 gpt-4o-mini）
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✅ 環境設定完成")
print(f"📦 LangGraph 已載入")
print(f"🤖 LLM 模型：gpt-4o-mini")

---
## Part 1：LangGraph 核心概念

### 1.1 什麼是 LangGraph？

LangGraph 是 LangChain 生態系統中的**有向圖狀態機框架**，專為構建多步驟、多智能體的 LLM 應用設計。

**核心概念：**

| 概念 | 說明 | 類比 |
|------|------|------|
| **StateGraph** | 圖的主體，管理全域狀態 | 工廠流水線的整體架構 |
| **State（狀態）** | 在節點間傳遞的共享資料結構 | 流水線上的工件托盤 |
| **Node（節點）** | 執行特定功能的函式 | 流水線上的工作站 |
| **Edge（邊）** | 節點間的連線，決定資料流向 | 工件托盤的傳送帶 |
| **Conditional Edge** | 根據狀態動態決定下一個節點 | 智慧分流閥門 |

### 1.2 LangGraph vs 傳統 Chain

```
傳統 Chain（線性）：
A → B → C → D （固定順序，無法回退）

LangGraph（圖形）：
        ┌─────────────────┐
        ↓                 │
A → B → C → D（成功）     │
        │                 │
        └──→ E（失敗重試）─┘
```

LangGraph 支援**循環**、**條件分支**、**並行執行**，是構建生產級 Agent 系統的關鍵工具。

In [ ]:
# ============================================================
# 1.3 TypedDict 狀態定義
# ============================================================
# LangGraph 使用 TypedDict 定義圖的共享狀態
# 每個節點可以讀取並更新這個狀態

class SimpleState(TypedDict):
    """簡單示範用的狀態結構"""
    input_text: str          # 使用者輸入
    processed_text: str      # 處理後的文字
    output_text: str         # 最終輸出
    step_log: List[str]      # 每個步驟的記錄

print("✅ 狀態結構定義完成")
print("\nSimpleState 欄位：")
for field, annotation in SimpleState.__annotations__.items():
    print(f"  • {field}: {annotation}")

In [ ]:
# ============================================================
# 1.4 簡單三節點圖示範：input → process → output
# ============================================================

def input_node(state: SimpleState) -> SimpleState:
    """節點1：接收輸入並記錄"""
    print(f"[input_node] 收到輸入：{state['input_text']}")
    log = state.get("step_log", [])
    return {
        **state,
        "step_log": log + [f"Step 1 - 輸入接收：{state['input_text']}"],
        "processed_text": "",
        "output_text": ""
    }

def process_node(state: SimpleState) -> SimpleState:
    """節點2：使用 LLM 處理文字"""
    print(f"[process_node] 正在處理文字...")
    
    response = llm.invoke([
        SystemMessage(content="你是一個文字分析助手，請簡要分析輸入文字的主題（用一句話）。"),
        HumanMessage(content=state["input_text"])
    ])
    
    processed = response.content
    log = state.get("step_log", [])
    print(f"[process_node] 分析結果：{processed}")
    
    return {
        **state,
        "processed_text": processed,
        "step_log": log + [f"Step 2 - LLM 分析：{processed[:50]}..."]
    }

def output_node(state: SimpleState) -> SimpleState:
    """節點3：格式化最終輸出"""
    print(f"[output_node] 格式化輸出...")
    
    output = f"📋 分析報告\n原始輸入：{state['input_text']}\n分析結果：{state['processed_text']}"
    log = state.get("step_log", [])
    
    return {
        **state,
        "output_text": output,
        "step_log": log + ["Step 3 - 輸出格式化完成"]
    }

# ── 建立圖 ──────────────────────────────────────────────────
simple_graph_builder = StateGraph(SimpleState)

# 加入節點
simple_graph_builder.add_node("input", input_node)
simple_graph_builder.add_node("process", process_node)
simple_graph_builder.add_node("output", output_node)

# 連接邊（線性流程）
simple_graph_builder.add_edge(START, "input")
simple_graph_builder.add_edge("input", "process")
simple_graph_builder.add_edge("process", "output")
simple_graph_builder.add_edge("output", END)

# 編譯圖
simple_graph = simple_graph_builder.compile()

print("✅ 簡單三節點圖建立完成")
print("圖結構：START → input → process → output → END")

In [ ]:
# ── 執行圖 ───────────────────────────────────────────────────
print("=" * 60)
print("執行簡單三節點圖")
print("=" * 60)

initial_state: SimpleState = {
    "input_text": "人工智慧正在改變醫療產業，從影像診斷到藥物研發都有重大突破。",
    "processed_text": "",
    "output_text": "",
    "step_log": []
}

result = simple_graph.invoke(initial_state)

print("\n" + "=" * 60)
print("最終結果：")
print(result["output_text"])
print("\n執行記錄：")
for log in result["step_log"]:
    print(f"  ✓ {log}")

---
## Part 2：Planner → Executor → Critic 模式

### 2.1 模式說明

這是多智能體系統中最經典的協作模式之一：

```
使用者查詢
     │
     ▼
┌─────────────┐
│   Planner   │  分解任務 → 產生子任務清單（JSON）
└──────┬──────┘
       │ 子任務清單
       ▼
┌─────────────┐
│  Executor   │  逐一執行子任務 → 產生結果
└──────┬──────┘
       │ 執行結果
       ▼
┌─────────────┐    評分 < 7     ┌─────────────┐
│   Critic    │ ─────────────→  │  重新執行   │
│   審查員    │                  └──────┬──────┘
└──────┬──────┘                        │
  評分 ≥ 7│                            ▼
       ▼  └────────────────────  Executor
     結束
```

**各角色職責：**
- **Planner**：理解使用者意圖，將複雜問題分解為可執行的子任務
- **Executor**：接收子任務，呼叫工具或 LLM 逐一完成
- **Critic**：審查所有執行結果，給予品質評分（1-10），決定 PASS 或 RETRY

In [ ]:
# ============================================================
# 2.2 定義 Planner → Executor → Critic 的狀態結構
# ============================================================

class PECState(TypedDict):
    """Planner-Executor-Critic 系統的共享狀態"""
    user_query: str                    # 使用者原始查詢
    subtasks: List[str]                # Planner 生成的子任務清單
    execution_results: List[str]       # Executor 的執行結果
    critic_score: int                  # Critic 給的評分（1-10）
    critic_feedback: str               # Critic 的具體回饋
    critic_decision: str               # "PASS" 或 "RETRY"
    retry_count: int                   # 重試次數
    final_answer: str                  # 最終整合答案

print("✅ PECState 狀態結構定義完成")

In [ ]:
# ============================================================
# 2.3 Planner Agent：分解任務
# ============================================================

def planner_agent(state: PECState) -> PECState:
    """
    Planner Agent：分析使用者查詢，生成子任務清單
    輸出：JSON 格式的子任務陣列
    """
    print("\n" + "─" * 50)
    print("🗂️  [Planner Agent] 開始分解任務...")
    print(f"   查詢：{state['user_query']}")
    
    system_prompt = """你是一個任務規劃專家。
將使用者的查詢分解為 3-5 個具體可執行的子任務。
必須以 JSON 陣列格式回應，每個元素是一個子任務字串。
範例：["子任務1", "子任務2", "子任務3"]
只回傳 JSON，不要加其他說明。"""
    
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"請分解以下查詢：{state['user_query']}")
    ])
    
    # 解析 JSON 回應
    try:
        # 清理可能的 markdown 代碼塊
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        subtasks = json.loads(raw)
    except json.JSONDecodeError:
        # 備援：直接用換行切分
        subtasks = [line.strip() for line in response.content.split("\n") if line.strip()]
    
    print(f"   ✅ 生成 {len(subtasks)} 個子任務：")
    for i, task in enumerate(subtasks, 1):
        print(f"      {i}. {task}")
    
    return {
        **state,
        "subtasks": subtasks,
        "execution_results": [],
        "retry_count": state.get("retry_count", 0)
    }

print("✅ Planner Agent 定義完成")

In [ ]:
# ============================================================
# 2.4 Executor Agent：執行子任務
# ============================================================

def executor_agent(state: PECState) -> PECState:
    """
    Executor Agent：逐一執行 Planner 產生的子任務
    每個子任務呼叫 LLM 產生具體結果
    """
    print("\n" + "─" * 50)
    retry = state.get("retry_count", 0)
    print(f"⚙️  [Executor Agent] 開始執行子任務（第 {retry + 1} 次）...")
    
    system_prompt = """你是一個任務執行專家。
針對給定的子任務，提供具體、詳細、準確的執行結果。
回應要包含實質內容，長度約 2-3 句話。"""
    
    results = []
    for i, task in enumerate(state["subtasks"], 1):
        print(f"   執行子任務 {i}/{len(state['subtasks'])}：{task[:40]}...")
        
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"請執行以下子任務並提供詳細結果：\n{task}")
        ])
        
        result = f"[子任務{i}] {task}\n結果：{response.content}"
        results.append(result)
        print(f"      ✓ 完成")
    
    print(f"   ✅ 所有 {len(results)} 個子任務執行完畢")
    
    return {
        **state,
        "execution_results": results
    }

print("✅ Executor Agent 定義完成")

In [ ]:
# ============================================================
# 2.5 Critic Agent：審查並評分
# ============================================================

def critic_agent(state: PECState) -> PECState:
    """
    Critic Agent：審查所有執行結果
    給予 1-10 評分，並決定 PASS（通過）或 RETRY（重試）
    """
    print("\n" + "─" * 50)
    print("🔍 [Critic Agent] 開始審查執行結果...")
    
    all_results = "\n\n".join(state["execution_results"])
    
    system_prompt = """你是一個嚴格的品質審查專家。
評估給定的任務執行結果，並以 JSON 格式回應：
{
  "score": <1-10的整數>,
  "feedback": "具體回饋說明",
  "decision": "PASS" 或 "RETRY"
}
評分標準：
- 8-10：優秀，資訊完整準確 → PASS
- 6-7：良好，基本達標 → PASS
- 1-5：不足，需要改進 → RETRY
只回傳 JSON，不要加其他說明。"""
    
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"原始查詢：{state['user_query']}\n\n執行結果：\n{all_results}")
    ])
    
    # 解析評審結果
    try:
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        review = json.loads(raw)
        score = int(review.get("score", 7))
        feedback = review.get("feedback", "無具體回饋")
        decision = review.get("decision", "PASS")
    except (json.JSONDecodeError, ValueError):
        score = 7
        feedback = response.content
        decision = "PASS"
    
    # 最多重試 2 次
    retry_count = state.get("retry_count", 0)
    if retry_count >= 2:
        decision = "PASS"
        print(f"   ⚠️  已達最大重試次數，強制 PASS")
    
    print(f"   📊 評分：{score}/10")
    print(f"   💬 回饋：{feedback[:80]}..." if len(feedback) > 80 else f"   💬 回饋：{feedback}")
    print(f"   🎯 決定：{decision}")
    
    return {
        **state,
        "critic_score": score,
        "critic_feedback": feedback,
        "critic_decision": decision,
        "retry_count": retry_count + (1 if decision == "RETRY" else 0)
    }

def synthesize_answer(state: PECState) -> PECState:
    """最終整合：將所有執行結果整合成完整答案"""
    print("\n" + "─" * 50)
    print("📝 [Synthesizer] 整合最終答案...")
    
    all_results = "\n\n".join(state["execution_results"])
    
    response = llm.invoke([
        SystemMessage(content="你是一個報告撰寫專家，請將多個子任務結果整合成一份結構清晰的完整回答。"),
        HumanMessage(content=f"原始問題：{state['user_query']}\n\n各子任務結果：\n{all_results}\n\n請整合成完整答案：")
    ])
    
    print("   ✅ 整合完成")
    return {
        **state,
        "final_answer": response.content
    }

print("✅ Critic Agent & Synthesizer 定義完成")

In [ ]:
# ============================================================
# 2.6 組裝 Planner → Executor → Critic 圖
# ============================================================

def route_after_critic(state: PECState) -> str:
    """條件路由：根據 Critic 的決定選擇下一個節點"""
    if state["critic_decision"] == "RETRY":
        print(f"   🔄 路由至：executor（重試，第 {state['retry_count']} 次）")
        return "executor"
    else:
        print(f"   ✅ 路由至：synthesize（PASS，評分 {state['critic_score']}/10）")
        return "synthesize"

# 建立圖
pec_builder = StateGraph(PECState)

# 加入節點
pec_builder.add_node("planner", planner_agent)
pec_builder.add_node("executor", executor_agent)
pec_builder.add_node("critic", critic_agent)
pec_builder.add_node("synthesize", synthesize_answer)

# 連接邊
pec_builder.add_edge(START, "planner")
pec_builder.add_edge("planner", "executor")
pec_builder.add_edge("executor", "critic")

# 條件邊：根據 Critic 決定路由
pec_builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {
        "executor": "executor",   # RETRY → 回到 executor
        "synthesize": "synthesize" # PASS → 進行整合
    }
)
pec_builder.add_edge("synthesize", END)

# 編譯
pec_graph = pec_builder.compile()

print("✅ Planner → Executor → Critic 圖建立完成")
print("""
圖結構：
START → planner → executor → critic
                      ↑          │
                      └── RETRY ─┘
                                 │
                          PASS   ↓
                         synthesize → END
""")

In [ ]:
# ── 執行 PEC 圖 ──────────────────────────────────────────────
print("=" * 60)
print("執行 Planner → Executor → Critic 多智能體系統")
print("=" * 60)

pec_initial: PECState = {
    "user_query": "請分析台灣電動車產業的發展現況、主要挑戰與未來機會",
    "subtasks": [],
    "execution_results": [],
    "critic_score": 0,
    "critic_feedback": "",
    "critic_decision": "",
    "retry_count": 0,
    "final_answer": ""
}

pec_result = pec_graph.invoke(pec_initial)

print("\n" + "=" * 60)
print("📊 執行摘要")
print("=" * 60)
print(f"子任務數量：{len(pec_result['subtasks'])}")
print(f"Critic 評分：{pec_result['critic_score']}/10")
print(f"重試次數：{pec_result['retry_count']}")
print(f"\n最終答案（前 500 字）：")
print(pec_result["final_answer"][:500] + "..." if len(pec_result["final_answer"]) > 500 else pec_result["final_answer"])

---
## Part 3：階層式委派（Hierarchical Delegation）

### 3.1 架構說明

階層式架構將 Agent 組織成**上下層級**，Orchestrator 負責理解任務並分派給最適合的專業子智能體：

```
               ┌─────────────────────┐
               │   Orchestrator      │  ← 大腦：理解意圖，決定路由
               │   （指揮官）         │
               └──────────┬──────────┘
                          │ 根據查詢類型路由
          ┌───────────────┼───────────────┐
          ▼               ▼               ▼
   ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
   │ Research    │ │ Calculator  │ │  Summary    │
   │ Agent       │ │ Agent       │ │  Agent      │
   │ （研究員）   │ │ （計算員）   │ │ （摘要員）   │
   └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
          └───────────────┴───────────────┘
                          │ 結果回傳
               ┌──────────┴──────────┐
               │   Final Synthesis   │  ← 整合所有子智能體結果
               └─────────────────────┘
```

**三個專業子智能體：**
- **ResearchAgent**：模擬 RAG 搜索，回答知識性問題
- **CalculatorAgent**：處理數學計算與數值分析
- **SummaryAgent**：摘要整理、重點提煉

In [ ]:
# ============================================================
# 3.2 定義階層式系統的狀態
# ============================================================

class HierarchicalState(TypedDict):
    """階層式多智能體系統的共享狀態"""
    user_query: str               # 使用者查詢
    query_type: str               # 查詢類型：research / calculation / summary
    orchestrator_plan: str        # Orchestrator 的分析與計畫
    specialist_result: str        # 專業子智能體的執行結果
    final_synthesis: str          # Orchestrator 的最終整合
    agent_trace: List[str]        # 智能體執行追蹤記錄

print("✅ HierarchicalState 定義完成")

In [ ]:
# ============================================================
# 3.3 Orchestrator Agent
# ============================================================

def orchestrator_agent(state: HierarchicalState) -> HierarchicalState:
    """
    Orchestrator：分析查詢類型，決定要呼叫哪個專業子智能體
    """
    print("\n" + "═" * 55)
    print("🎯 [Orchestrator] 分析查詢...")
    print(f"   查詢：{state['user_query']}")
    
    system_prompt = """你是一個智能路由指揮官。分析使用者查詢，判斷最適合的處理方式。
以 JSON 格式回應：
{
  "query_type": "research" | "calculation" | "summary",
  "plan": "你的分析與計畫說明"
}
- research：需要知識搜索、事實查詢、背景研究
- calculation：需要數學計算、數值分析、統計
- summary：需要摘要、整理、重點提煉
只回傳 JSON。"""
    
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=state["user_query"])
    ])
    
    try:
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        parsed = json.loads(raw)
        query_type = parsed.get("query_type", "research")
        plan = parsed.get("plan", "")
    except json.JSONDecodeError:
        query_type = "research"
        plan = response.content
    
    trace = state.get("agent_trace", [])
    print(f"   🗺️  查詢類型：{query_type}")
    print(f"   📋 計畫：{plan[:80]}..." if len(plan) > 80 else f"   📋 計畫：{plan}")
    
    return {
        **state,
        "query_type": query_type,
        "orchestrator_plan": plan,
        "agent_trace": trace + [f"Orchestrator → 路由至 {query_type} 智能體"]
    }

print("✅ Orchestrator Agent 定義完成")

In [ ]:
# ============================================================
# 3.4 三個專業子智能體
# ============================================================

def research_agent(state: HierarchicalState) -> HierarchicalState:
    """ResearchAgent：模擬 RAG 搜索，回答知識性問題"""
    print("\n" + "─" * 55)
    print("🔬 [ResearchAgent] 執行知識搜索...")
    
    response = llm.invoke([
        SystemMessage(content="""你是一個研究專家，模擬 RAG 知識庫搜索。
提供詳細、有據可查的資訊，包含具體事實和數據。回應 200-300 字。"""),
        HumanMessage(content=f"研究查詢：{state['user_query']}\n計畫：{state['orchestrator_plan']}")
    ])
    
    trace = state.get("agent_trace", [])
    print(f"   ✅ 研究完成，結果長度：{len(response.content)} 字元")
    
    return {
        **state,
        "specialist_result": f"[ResearchAgent 結果]\n{response.content}",
        "agent_trace": trace + ["ResearchAgent → 知識搜索完成"]
    }


def calculator_agent(state: HierarchicalState) -> HierarchicalState:
    """CalculatorAgent：處理數學計算與數值分析"""
    print("\n" + "─" * 55)
    print("🧮 [CalculatorAgent] 執行數值計算...")
    
    response = llm.invoke([
        SystemMessage(content="""你是一個數學計算專家。
提供精確的計算過程、公式說明和數值結果。用條列方式呈現計算步驟。"""),
        HumanMessage(content=f"計算任務：{state['user_query']}\n計畫：{state['orchestrator_plan']}")
    ])
    
    trace = state.get("agent_trace", [])
    print(f"   ✅ 計算完成")
    
    return {
        **state,
        "specialist_result": f"[CalculatorAgent 結果]\n{response.content}",
        "agent_trace": trace + ["CalculatorAgent → 數值計算完成"]
    }


def summary_agent(state: HierarchicalState) -> HierarchicalState:
    """SummaryAgent：摘要整理、重點提煉"""
    print("\n" + "─" * 55)
    print("📄 [SummaryAgent] 執行摘要整理...")
    
    response = llm.invoke([
        SystemMessage(content="""你是一個摘要整理專家。
提供結構清晰的摘要，包含：核心重點、關鍵洞察、行動建議。"""),
        HumanMessage(content=f"摘要任務：{state['user_query']}\n計畫：{state['orchestrator_plan']}")
    ])
    
    trace = state.get("agent_trace", [])
    print(f"   ✅ 摘要完成")
    
    return {
        **state,
        "specialist_result": f"[SummaryAgent 結果]\n{response.content}",
        "agent_trace": trace + ["SummaryAgent → 摘要整理完成"]
    }


def final_synthesis_node(state: HierarchicalState) -> HierarchicalState:
    """Orchestrator 整合子智能體結果"""
    print("\n" + "─" * 55)
    print("🎯 [Orchestrator] 整合子智能體結果...")
    
    response = llm.invoke([
        SystemMessage(content="你是一個整合報告專家，將專業子智能體的結果整合成最終完整回答。"),
        HumanMessage(content=f"原始問題：{state['user_query']}\n子智能體結果：{state['specialist_result']}")
    ])
    
    trace = state.get("agent_trace", [])
    print("   ✅ 最終整合完成")
    
    return {
        **state,
        "final_synthesis": response.content,
        "agent_trace": trace + ["Orchestrator → 最終整合完成"]
    }

print("✅ 三個專業子智能體定義完成")

In [ ]:
# ============================================================
# 3.5 組裝階層式圖（含條件路由）
# ============================================================

def route_to_specialist(state: HierarchicalState) -> str:
    """條件路由函式：根據查詢類型選擇專業子智能體"""
    query_type = state.get("query_type", "research")
    route_map = {
        "research": "research_agent",
        "calculation": "calculator_agent",
        "summary": "summary_agent"
    }
    target = route_map.get(query_type, "research_agent")
    print(f"   🔀 路由決定：{query_type} → {target}")
    return target

# 建立圖
hier_builder = StateGraph(HierarchicalState)

# 加入節點
hier_builder.add_node("orchestrator", orchestrator_agent)
hier_builder.add_node("research_agent", research_agent)
hier_builder.add_node("calculator_agent", calculator_agent)
hier_builder.add_node("summary_agent", summary_agent)
hier_builder.add_node("final_synthesis", final_synthesis_node)

# 邊：START → orchestrator
hier_builder.add_edge(START, "orchestrator")

# 條件邊：orchestrator → 三個專業子智能體之一
hier_builder.add_conditional_edges(
    "orchestrator",
    route_to_specialist,
    {
        "research_agent": "research_agent",
        "calculator_agent": "calculator_agent",
        "summary_agent": "summary_agent"
    }
)

# 所有子智能體都流向 final_synthesis
hier_builder.add_edge("research_agent", "final_synthesis")
hier_builder.add_edge("calculator_agent", "final_synthesis")
hier_builder.add_edge("summary_agent", "final_synthesis")
hier_builder.add_edge("final_synthesis", END)

# 編譯
hier_graph = hier_builder.compile()

print("✅ 階層式圖建立完成")

In [ ]:
# ── 測試三種不同類型的查詢 ──────────────────────────────────

test_queries = [
    "量子運算的基本原理是什麼？與傳統電腦有何不同？",
    "如果我每月投資 1 萬元，年利率 6%，20 年後本利和是多少？",
    "請整理 GPT-4 的主要能力和限制重點"
]

for i, query in enumerate(test_queries, 1):
    print("\n" + "═" * 60)
    print(f"測試 {i}/3")
    print("═" * 60)
    
    initial: HierarchicalState = {
        "user_query": query,
        "query_type": "",
        "orchestrator_plan": "",
        "specialist_result": "",
        "final_synthesis": "",
        "agent_trace": []
    }
    
    result = hier_graph.invoke(initial)
    
    print(f"\n📋 智能體執行路徑：")
    for step in result["agent_trace"]:
        print(f"   → {step}")
    print(f"\n💡 最終答案（前 300 字）：")
    print(result["final_synthesis"][:300] + "..." if len(result["final_synthesis"]) > 300 else result["final_synthesis"])

---
## Part 4：Human-in-the-Loop（人工審查檢查點）

### 4.1 為什麼需要 Human-in-the-Loop？

在高風險場景中（如醫療診斷、法律建議、金融交易），AI 的決策需要人工確認：

```
自動執行 → 暫停等待人工審查 → 人工批准/拒絕 → 繼續/停止
```

LangGraph 提供 `interrupt_before` 機制，讓圖在執行到特定節點前**暫停**，等待外部輸入。

**關鍵組件：**
- `MemorySaver`：儲存圖的執行狀態（快照）
- `interrupt_before`：指定在哪些節點前暫停
- `thread_id`：每次對話的唯一識別碼
- `Command(resume=...)`：恢復執行並傳入人工決定

In [ ]:
# ============================================================
# 4.2 定義帶有人工審查的狀態與節點
# ============================================================
from langgraph.types import Command, interrupt

class HITLState(TypedDict):
    """Human-in-the-Loop 系統的狀態"""
    user_request: str          # 使用者請求
    ai_proposal: str           # AI 的提案
    human_decision: str        # 人工決定："approve" 或 "reject"
    human_comment: str         # 人工的意見
    final_action: str          # 最終執行動作
    execution_log: List[str]   # 執行記錄


def generate_proposal(state: HITLState) -> HITLState:
    """AI 生成提案（自動執行）"""
    print("\n" + "─" * 55)
    print("🤖 [AI] 生成處理提案...")
    
    response = llm.invoke([
        SystemMessage(content="你是一個業務決策助手。請針對使用者請求，提出一個具體的行動建議方案，包含：行動項目、預期效果、潛在風險。"),
        HumanMessage(content=f"請求：{state['user_request']}")
    ])
    
    log = state.get("execution_log", [])
    print(f"   📋 提案已生成（{len(response.content)} 字元）")
    
    return {
        **state,
        "ai_proposal": response.content,
        "execution_log": log + ["AI 提案生成完成"]
    }


def human_review_node(state: HITLState) -> HITLState:
    """
    人工審查節點（使用 interrupt）
    圖會在此暫停，等待人工輸入
    """
    print("\n" + "═" * 55)
    print("🛑 [Human Review] 等待人工審查...")
    print("\nAI 提案：")
    print(state["ai_proposal"][:300] + "..." if len(state["ai_proposal"]) > 300 else state["ai_proposal"])
    print("\n" + "═" * 55)
    
    # interrupt() 會暫停圖的執行，返回一個 value 給外部
    # 當圖恢復時，interrupt() 會返回人工輸入的值
    human_input = interrupt({
        "proposal": state["ai_proposal"],
        "message": "請審查 AI 提案並決定是否批准（approve/reject）"
    })
    
    # 解析人工輸入
    if isinstance(human_input, dict):
        decision = human_input.get("decision", "approve")
        comment = human_input.get("comment", "")
    else:
        decision = str(human_input)
        comment = ""
    
    log = state.get("execution_log", [])
    print(f"   ✅ 人工決定：{decision}")
    if comment:
        print(f"   💬 人工意見：{comment}")
    
    return {
        **state,
        "human_decision": decision,
        "human_comment": comment,
        "execution_log": log + [f"人工審查：{decision}"]
    }


def execute_approved(state: HITLState) -> HITLState:
    """執行已批准的提案"""
    print("\n" + "─" * 55)
    print("✅ [Executor] 執行已批准的提案...")
    
    response = llm.invoke([
        SystemMessage(content="你是執行專家，請描述如何具體實施這個提案。"),
        HumanMessage(content=f"提案：{state['ai_proposal']}\n人工意見：{state['human_comment']}")
    ])
    
    log = state.get("execution_log", [])
    return {
        **state,
        "final_action": f"✅ 已執行\n{response.content}",
        "execution_log": log + ["提案執行完成"]
    }


def handle_rejected(state: HITLState) -> HITLState:
    """處理被拒絕的提案"""
    print("\n" + "─" * 55)
    print("❌ [Handler] 處理被拒絕的提案...")
    
    log = state.get("execution_log", [])
    return {
        **state,
        "final_action": f"❌ 提案已拒絕\n原因：{state['human_comment']}\n提案已歸檔，不執行任何操作。",
        "execution_log": log + ["提案拒絕，已歸檔"]
    }

print("✅ HITL 節點定義完成")

In [ ]:
# ============================================================
# 4.3 組裝 HITL 圖
# ============================================================

def route_after_review(state: HITLState) -> str:
    """根據人工決定路由"""
    decision = state.get("human_decision", "approve")
    if decision.lower() == "approve":
        return "execute"
    else:
        return "reject"

# 使用 MemorySaver 作為 checkpointer（支援 interrupt）
memory = MemorySaver()

hitl_builder = StateGraph(HITLState)

# 加入節點
hitl_builder.add_node("generate_proposal", generate_proposal)
hitl_builder.add_node("human_review", human_review_node)
hitl_builder.add_node("execute_approved", execute_approved)
hitl_builder.add_node("handle_rejected", handle_rejected)

# 邊
hitl_builder.add_edge(START, "generate_proposal")
hitl_builder.add_edge("generate_proposal", "human_review")
hitl_builder.add_conditional_edges(
    "human_review",
    route_after_review,
    {
        "execute": "execute_approved",
        "reject": "handle_rejected"
    }
)
hitl_builder.add_edge("execute_approved", END)
hitl_builder.add_edge("handle_rejected", END)

# 編譯時加入 checkpointer（HITL 必需）
hitl_graph = hitl_builder.compile(
    checkpointer=memory
)

print("✅ HITL 圖建立完成（含 MemorySaver checkpointer）")
print("""
圖結構：
START → generate_proposal → human_review → [暫停等待人工]
                                │
                    approve ────┼──── reject
                         │               │
                 execute_approved   handle_rejected
                         │               │
                        END            END
""")

In [ ]:
# ============================================================
# 4.4 模擬 Human-in-the-Loop 執行流程
# ============================================================
from langgraph.types import Command

# 使用 thread_id 追蹤對話
thread_config = {"configurable": {"thread_id": "hitl-demo-001"}}

print("=" * 60)
print("📋 Human-in-the-Loop 執行示範")
print("=" * 60)

initial_hitl: HITLState = {
    "user_request": "我想在台北市中山區開一家手搖飲料店，請提供創業計畫建議",
    "ai_proposal": "",
    "human_decision": "",
    "human_comment": "",
    "final_action": "",
    "execution_log": []
}

# ── 第一階段：執行到 interrupt 點 ────────────────────────────
print("\n[階段 1] 啟動圖，執行至人工審查點...")

result_phase1 = None
for event in hitl_graph.stream(initial_hitl, thread_config, stream_mode="values"):
    # 當遇到 interrupt 時，stream 會停止
    result_phase1 = event

print("\n[圖已暫停，等待人工審查]")

# 取得當前狀態快照
snapshot = hitl_graph.get_state(thread_config)
print(f"\n圖當前狀態：{snapshot.next}")

In [ ]:
# ── 第二階段：模擬人工輸入，恢復執行 ───────────────────────
print("=" * 60)
print("[階段 2] 模擬人工審查並輸入決定...")
print("=" * 60)

# 模擬人工決定（批准）
human_approval = {
    "decision": "approve",
    "comment": "計畫可行，請特別注意衛生許可證申請流程"
}

print(f"\n人工決定：{human_approval['decision']}")
print(f"人工意見：{human_approval['comment']}")
print("\n恢復圖執行...")

# 使用 Command(resume=...) 恢復執行
final_result = hitl_graph.invoke(
    Command(resume=human_approval),
    thread_config
)

print("\n" + "=" * 60)
print("📊 執行記錄：")
for log in final_result["execution_log"]:
    print(f"  ✓ {log}")

print(f"\n最終動作（前 400 字）：")
print(final_result["final_action"][:400])

In [ ]:
# ── 示範拒絕情境 ─────────────────────────────────────────────
print("=" * 60)
print("📋 示範：人工拒絕提案")
print("=" * 60)

thread_config_reject = {"configurable": {"thread_id": "hitl-demo-002"}}

# 重新執行到暫停點
for event in hitl_graph.stream(initial_hitl, thread_config_reject, stream_mode="values"):
    pass

# 模擬人工拒絕
human_rejection = {
    "decision": "reject",
    "comment": "預算評估過於樂觀，需要重新評估資金需求"
}

print(f"\n人工決定：{human_rejection['decision']}")
print(f"人工意見：{human_rejection['comment']}")

rejected_result = hitl_graph.invoke(
    Command(resume=human_rejection),
    thread_config_reject
)

print(f"\n最終動作：")
print(rejected_result["final_action"])

---
## Part 5：並行子智能體（Parallel Execution）

### 5.1 為什麼需要並行執行？

當多個子任務**彼此獨立**時，串行執行浪費時間：

```
串行（Sequential）：                    並行（Parallel）：
任務A (3秒) → 任務B (2秒) → 任務C (4秒)  任務A (3秒) ┐
總計：9秒                                任務B (2秒) ├→ 結果整合
                                         任務C (4秒) ┘
                                         總計：4秒（最慢任務的時間）
```

LangGraph 結合 Python 的 `asyncio` 實現並行子智能體執行。

In [ ]:
# ============================================================
# 5.2 定義並行執行的狀態與異步節點
# ============================================================

class ParallelState(TypedDict):
    """並行執行系統的狀態"""
    topic: str                        # 研究主題
    research_result: str              # 研究子智能體結果
    analysis_result: str              # 分析子智能體結果
    creative_result: str              # 創意子智能體結果
    parallel_results: List[str]       # 所有並行結果
    final_report: str                 # 最終整合報告


# ── 異步子智能體函式 ─────────────────────────────────────────

async def async_research_agent(topic: str) -> str:
    """異步研究子智能體"""
    from langchain_openai import ChatOpenAI
    async_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    
    response = await async_llm.ainvoke([
        SystemMessage(content="你是研究專家，提供深入的背景知識和事實資訊。"),
        HumanMessage(content=f"研究主題：{topic}\n請提供關鍵事實和現況（150字以內）")
    ])
    return f"[研究結果] {response.content}"


async def async_analysis_agent(topic: str) -> str:
    """異步分析子智能體"""
    from langchain_openai import ChatOpenAI
    async_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    
    response = await async_llm.ainvoke([
        SystemMessage(content="你是數據分析專家，擅長識別趨勢和洞察。"),
        HumanMessage(content=f"分析主題：{topic}\n請提供趨勢分析和關鍵洞察（150字以內）")
    ])
    return f"[分析結果] {response.content}"


async def async_creative_agent(topic: str) -> str:
    """異步創意子智能體"""
    from langchain_openai import ChatOpenAI
    async_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    
    response = await async_llm.ainvoke([
        SystemMessage(content="你是創意策略師，擅長提出創新解決方案。"),
        HumanMessage(content=f"創意主題：{topic}\n請提供 3 個創新應用或解決方案（150字以內）")
    ])
    return f"[創意方案] {response.content}"


print("✅ 異步子智能體函式定義完成")

In [ ]:
# ============================================================
# 5.3 並行執行節點（asyncio.gather）
# ============================================================

async def parallel_execution_node(state: ParallelState) -> ParallelState:
    """
    並行執行節點：同時啟動所有子智能體
    使用 asyncio.gather() 並行等待所有結果
    """
    print("\n" + "═" * 55)
    print(f"⚡ [並行執行] 同時啟動 3 個子智能體...")
    print(f"   主題：{state['topic']}")
    
    start_time = time.time()
    
    # asyncio.gather：並行執行所有任務，等待全部完成
    research, analysis, creative = await asyncio.gather(
        async_research_agent(state["topic"]),
        async_analysis_agent(state["topic"]),
        async_creative_agent(state["topic"])
    )
    
    elapsed = time.time() - start_time
    print(f"   ✅ 所有子智能體完成！並行耗時：{elapsed:.2f} 秒")
    
    return {
        **state,
        "research_result": research,
        "analysis_result": analysis,
        "creative_result": creative,
        "parallel_results": [research, analysis, creative]
    }


async def sequential_execution_node(state: ParallelState) -> ParallelState:
    """
    串行執行節點（對照組）：依序執行所有子智能體
    """
    print("\n" + "─" * 55)
    print(f"🐌 [串行執行] 依序執行 3 個子智能體...")
    
    start_time = time.time()
    
    research = await async_research_agent(state["topic"])
    print("   ✓ 研究子智能體完成")
    
    analysis = await async_analysis_agent(state["topic"])
    print("   ✓ 分析子智能體完成")
    
    creative = await async_creative_agent(state["topic"])
    print("   ✓ 創意子智能體完成")
    
    elapsed = time.time() - start_time
    print(f"   ⏱️  串行耗時：{elapsed:.2f} 秒")
    
    return {
        **state,
        "research_result": research,
        "analysis_result": analysis,
        "creative_result": creative,
        "parallel_results": [research, analysis, creative]
    }


async def synthesize_parallel_results(state: ParallelState) -> ParallelState:
    """整合所有並行結果"""
    print("\n" + "─" * 55)
    print("📊 [Synthesizer] 整合並行結果...")
    
    from langchain_openai import ChatOpenAI
    async_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    
    all_results = "\n\n".join(state["parallel_results"])
    
    response = await async_llm.ainvoke([
        SystemMessage(content="整合多個專業觀點，輸出結構化的完整報告。"),
        HumanMessage(content=f"主題：{state['topic']}\n\n各智能體結果：\n{all_results}")
    ])
    
    print("   ✅ 整合完成")
    return {
        **state,
        "final_report": response.content
    }

print("✅ 並行執行節點定義完成")

In [ ]:
# ============================================================
# 5.4 建立並行執行圖
# ============================================================

parallel_builder = StateGraph(ParallelState)

parallel_builder.add_node("parallel_execute", parallel_execution_node)
parallel_builder.add_node("synthesize", synthesize_parallel_results)

parallel_builder.add_edge(START, "parallel_execute")
parallel_builder.add_edge("parallel_execute", "synthesize")
parallel_builder.add_edge("synthesize", END)

parallel_graph = parallel_builder.compile()

print("✅ 並行執行圖建立完成")

In [ ]:
# ============================================================
# 5.5 並行 vs 串行效能比較
# ============================================================

topic = "生成式 AI 對未來工作市場的影響"

print("=" * 60)
print("⚡ 並行 vs 串行效能比較")
print("=" * 60)
print(f"主題：{topic}")

# ── 並行執行 ─────────────────────────────────────────────────
print("\n--- 並行執行（asyncio.gather）---")
t0 = time.time()

parallel_initial: ParallelState = {
    "topic": topic,
    "research_result": "",
    "analysis_result": "",
    "creative_result": "",
    "parallel_results": [],
    "final_report": ""
}

parallel_result = await parallel_graph.ainvoke(parallel_initial)
parallel_time = time.time() - t0

# ── 串行執行 ─────────────────────────────────────────────────
print("\n--- 串行執行（依序執行）---")
t1 = time.time()

# 直接呼叫串行版本
seq_state: ParallelState = {
    "topic": topic,
    "research_result": "",
    "analysis_result": "",
    "creative_result": "",
    "parallel_results": [],
    "final_report": ""
}
seq_state = await sequential_execution_node(seq_state)
sequential_time = time.time() - t1

# ── 結果比較 ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 效能比較結果")
print("=" * 60)
print(f"並行執行總耗時：{parallel_time:.2f} 秒")
print(f"串行執行總耗時：{sequential_time:.2f} 秒")
speedup = sequential_time / parallel_time if parallel_time > 0 else 0
print(f"速度提升比：{speedup:.2f}x")
print(f"節省時間：{sequential_time - parallel_time:.2f} 秒")

print(f"\n並行執行最終報告（前 400 字）：")
print(parallel_result["final_report"][:400] + "...")

---
## Part 6：面試 Q&A

以下整理了多智能體系統與 LangGraph 相關的核心面試題目，並提供深入解答。

### Q1：請解釋 LangGraph 的核心架構，它與傳統 LangChain Chain 有何本質差異？

**A：**

LangGraph 與傳統 Chain 的根本差異在於**執行模型**：

| 面向 | 傳統 Chain | LangGraph |
|------|-----------|----------|
| 執行流程 | 線性（DAG，無環） | 圖形（可含環路） |
| 狀態管理 | 每次呼叫獨立傳遞 | 全域共享狀態（TypedDict） |
| 條件分支 | 有限（RouterChain） | 任意條件邊（conditional_edges） |
| 迴圈支援 | 不支援 | 原生支援（RETRY 模式） |
| 持久化 | 需自行實作 | 內建 Checkpointer |
| HITL | 複雜 | 原生 interrupt() |

**核心架構三要素：**

1. **StateGraph**：圖的容器，持有全域狀態。狀態用 `TypedDict` 定義，確保型別安全。

2. **Nodes（節點）**：純函式，接受 state → 返回更新後的 state。每個節點負責單一職責（SRP）。

3. **Edges（邊）**：
   - 固定邊：`add_edge(A, B)` — A 執行後必定到 B
   - 條件邊：`add_conditional_edges(A, router_fn, mapping)` — 根據狀態動態選擇

**實務建議**：當任務需要重試邏輯、動態路由、人工介入時，優先選擇 LangGraph 而非傳統 Chain。

### Q2：Planner-Executor-Critic 模式的優缺點是什麼？何時應該使用？

**A：**

**優點：**
- **品質保障**：Critic 提供自動化的品質把關，避免低品質輸出
- **可解釋性**：三個角色各自獨立，執行過程透明易除錯
- **靈活性**：可以針對不同子任務使用不同 Executor 工具
- **可調整的品質標準**：調整 Critic 的閾值即可改變整體品質要求

**缺點：**
- **成本較高**：至少需要 3 次 LLM 呼叫，RETRY 會額外增加費用
- **延遲增加**：串行執行導致回應時間較長
- **可能無限循環**：需要設定最大重試次數（max_retries）作為安全閥
- **Critic 可能偏誤**：LLM 作為 Critic 可能不夠客觀

**適合場景：**
- 需要高品質輸出的長文生成（報告、程式碼）
- 有明確評分標準的任務（測試通過率、法規符合性）
- 允許較長處理時間的批次作業
- 成本允許的高價值任務

**不適合場景：**
- 即時互動（< 2 秒回應要求）
- 成本敏感的高頻場景
- 評分標準模糊、主觀性強的任務

### Q3：階層式（Hierarchical）vs 扁平式（Flat）多智能體架構如何選擇？

**A：**

```
階層式架構：                    扁平式架構：
Orchestrator                   Agent A ←→ Agent B
    ├── SubAgent A                  ↕           ↕
    ├── SubAgent B              Agent C ←→ Agent D
    └── SubAgent C
```

**選擇階層式的情境：**
- 任務有明確的**分工界線**（研究 vs 計算 vs 摘要）
- 需要**集中協調**，避免子智能體之間的衝突
- 子任務之間**相互獨立**，不需要頻繁溝通
- 系統需要**清晰的責任歸屬**（方便除錯）

**選擇扁平式的情境：**
- 任務需要**密集協作**（如程式碼審查 + 測試 + 修復的反覆迭代）
- 沒有明確的「指揮官」角色
- 所有智能體地位**平等**，需要民主決策
- 任務類型**高度同質**

**決策框架：**

| 問題 | 是 → | 否 → |
|------|------|------|
| 任務有清楚的分類? | 階層式 | 扁平式 |
| 子任務需要頻繁溝通? | 扁平式 | 階層式 |
| 需要中央協調和仲裁? | 階層式 | 扁平式 |
| 子任務執行時間差異大? | 並行+階層式 | 串行即可 |

**實務建議**：大多數生產系統使用**混合架構** — 頂層採用階層式（Orchestrator 協調），底層子智能體之間可以有扁平式的協作。

### Q4：LangGraph 的狀態管理（State Management）有哪些最佳實踐？

**A：**

**1. 使用 TypedDict 確保型別安全**
```python
class MyState(TypedDict):
    messages: List[str]      # 明確型別
    score: int               # 避免 Any
    is_complete: bool        # 語義清晰的命名
```

**2. 節點應返回完整狀態（不要只返回部分欄位）**
```python
# ❌ 錯誤：可能遺失其他欄位
def bad_node(state): return {"score": 10}

# ✅ 正確：使用 **state 保留其他欄位
def good_node(state): return {**state, "score": 10}
```

**3. 使用 Annotated 配合 operator.add 實現狀態累積**
```python
from typing import Annotated
import operator

class AccumulatingState(TypedDict):
    # 每個節點可以 append，不需要保留完整歷史
    messages: Annotated[List[str], operator.add]
```

**4. 善用 Checkpointer 進行狀態快照**
- 使用 `MemorySaver` 進行開發測試
- 生產環境使用 `SqliteSaver` 或 `PostgresSaver` 持久化
- 每個對話使用唯一的 `thread_id`

**5. 避免在狀態中存放大型物件**
- 狀態會在每個節點間傳遞，過大會影響效能
- 大型資料（如向量索引）應存於外部，狀態只保存引用

**6. 設計明確的終止條件**
- 設置 `max_iterations` 防止無限迴圈
- 在條件路由中加入「安全閥」分支
```python
def safe_router(state):
    if state["retry_count"] >= MAX_RETRIES:
        return "force_end"  # 強制終止
    return "retry" if state["needs_retry"] else "complete"
```

---
## 總結

### 本筆記本涵蓋的核心知識

| Part | 主題 | 關鍵技術 |
|------|------|----------|
| 1 | LangGraph 核心概念 | StateGraph, TypedDict, 節點, 邊 |
| 2 | Planner-Executor-Critic | 三層協作, 條件邊, 品質控制 |
| 3 | 階層式委派 | Orchestrator, 專業子智能體, 動態路由 |
| 4 | Human-in-the-Loop | interrupt(), MemorySaver, Command(resume) |
| 5 | 並行執行 | asyncio.gather(), ainvoke(), 效能比較 |

### 多智能體設計原則

1. **單一職責**：每個智能體只做一件事，做好一件事
2. **明確狀態**：用 TypedDict 明確定義資料結構
3. **防禦性程式設計**：所有 JSON 解析加 try-except，設置最大重試次數
4. **可觀察性**：在每個節點加入追蹤記錄（trace log）
5. **漸進式複雜度**：從單節點圖開始，逐步加入更多節點和條件

### 下一步學習資源

- [LangGraph 官方文件](https://langchain-ai.github.io/langgraph/)
- [LangGraph 教學系列](https://github.com/langchain-ai/langgraph/tree/main/examples)
- LangSmith：監控和除錯多智能體系統的追蹤工具